# Lab 6 :

**Prepared by:** Priyanka Boowade

 CSBS, 3rd year https://github.com/PriyankaBoowade/MachineLearning2026.git



In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the ABT using the corrected relative path
input_path = Path("../data/processed/olist_orders_abt.csv")
df = pd.read_csv(input_path)
print("Shape:", df.shape)
df.head()

Shape: (99441, 29)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017,11,18,5,...,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018,2,13,1,...,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery


In [6]:
target = "is_late_delivery"

identifier_columns = [
    "order_id", "customer_id", "customer_unique_id"
]

leakage_columns = [
    "delivery_days", "delivery_delay_days", "review_score",
    "review_comment_count", "has_review_comment", "is_low_review"
]

columns_to_drop = [
    c for c in identifier_columns + leakage_columns
    if c in df.columns
]
feature_df = df.drop(columns=columns_to_drop)
print("Safe features available:", feature_df.columns.tolist())

Safe features available: ['customer_city', 'customer_state', 'order_status', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'is_late_delivery', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category']


In [7]:
# Part C: Numerical Feature Engineering
if {"total_price", "total_items"}.issubset(df.columns):
    df["average_item_price"] = df["total_price"] / df["total_items"].replace(0, np.nan)

if {"total_freight", "total_price"}.issubset(df.columns):
    df["freight_ratio"] = df["total_freight"] / df["total_price"].replace(0, np.nan)

if {"total_items", "unique_sellers"}.issubset(df.columns):
    df["items_per_seller"] = df["total_items"] / df["unique_sellers"].replace(0, np.nan)

if {"unique_sellers", "total_items"}.issubset(df.columns):
    df["seller_diversity"] = df["unique_sellers"] / df["total_items"].replace(0, np.nan)

# Part D: Date and Time Features
if "order_day_of_week" in df.columns:
    df["is_weekend"] = (df["order_day_of_week"] >= 5).astype(int)

if "order_hour" in df.columns:
    df["is_business_hour"] = (df["order_hour"].between(9, 18)).astype(int)

if "order_month" in df.columns:
    df["is_year_end"] = (df["order_month"].isin([11, 12])).astype(int)

# Part E: Cyclical Features
if "order_month" in df.columns:
    df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12)

if "order_hour" in df.columns:
    df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24)

# Part F, G, H: Transformations, Binning, and Interactions
if "total_price" in df.columns:
    df["log_total_price"] = np.log1p(df["total_price"].clip(lower=0))
    df["price_band"] = pd.cut(
        df["total_price"],
        bins=[-np.inf, 100, 500, 1000, 5000, np.inf],
        labels=["very_low", "low", "medium", "high", "very_high"]
    )

if "total_freight" in df.columns:
    df["log_total_freight"] = np.log1p(df["total_freight"].clip(lower=0))

if {"total_items", "total_price"}.issubset(df.columns):
    df["items_x_price"] = df["total_items"] * df["total_price"]

In [8]:
# Variance-Based Selection
from sklearn.feature_selection import VarianceThreshold
numeric_df = df.select_dtypes(include=np.number).copy()
numeric_df = numeric_df.drop(columns=[target], errors="ignore")

selector = VarianceThreshold(threshold=0.0)
selector.fit(numeric_df.fillna(numeric_df.median()))
selected_numeric = numeric_df.columns[selector.get_support()]
print("Original numeric features:", len(numeric_df.columns), "| After variance filtering:", len(selected_numeric))

# Correlation-Based Selection
corr = numeric_df.corr(numeric_only=True)
threshold = 0.90
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        value = upper.loc[row, col]
        if pd.notna(value) and abs(value) > threshold:
            high_corr_pairs.append((row, col, value))
print("Highly correlated pairs (consider dropping one of each):", high_corr_pairs[:5])

# Mutual Information
from sklearn.feature_selection import mutual_info_classif
mi_df = numeric_df.fillna(numeric_df.median())
mi_scores = mutual_info_classif(mi_df, df[target], random_state=42)
mi_results = pd.DataFrame({
    "feature": mi_df.columns,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)
print("\nTop Mutual Information Scores:\n", mi_results.head(10))

# Model-Based Feature Importance
from sklearn.ensemble import RandomForestClassifier
X = mi_df
y = df[target]

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced")
rf.fit(X, y)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print("\nTop Random Forest Importance:\n", importance.head(10))

Original numeric features: 34 | After variance filtering: 34
Highly correlated pairs (consider dropping one of each): [('review_comment_count', 'has_review_comment', np.float64(0.9970565301210799)), ('total_payment_value', 'total_price', np.float64(0.9959698733878193)), ('total_payment_value', 'average_item_price', np.float64(0.9211962096778621)), ('total_price', 'average_item_price', np.float64(0.9331693361267265)), ('total_items', 'items_per_seller', np.float64(0.9573595867372893))]

Top Mutual Information Scores:
                 feature  mutual_information
7   delivery_delay_days            0.243227
5         delivery_days            0.133898
8          review_score            0.052955
9         is_low_review            0.041675
19       unique_sellers            0.027428
18      unique_products            0.023377
25     is_business_hour            0.017332
14  payment_types_count            0.016252
23     seller_diversity            0.014488
1           order_month            0.

In [9]:
candidate_features = [
    "total_price", "total_freight", "total_items",
    "unique_products", "unique_sellers", "order_month",
    "order_day_of_week", "order_hour", "average_item_price",
    "freight_ratio", "items_per_seller", "seller_diversity",
    "is_weekend", "is_business_hour", "month_sin",
    "month_cos", "hour_sin", "hour_cos", "log_total_price",
    "log_total_freight", "items_x_price"
]

final_features = [c for c in candidate_features if c in df.columns]
final_df = df[final_features + [target]].copy()
print("Final shape:", final_df.shape)

# UPDATED PATH: Pointing back up to the processed folder
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "olist_orders_feature_engineered.csv"

final_df.to_csv(output_path, index=False)
print("Successfully saved to:", output_path)

Final shape: (99441, 22)
Successfully saved to: ..\data\processed\olist_orders_feature_engineered.csv
